In [ ]:
import sagemaker
from sagemaker.session import Session
import boto3
import pandas as pd
from sklearn.model_selection import train_test_split

# create session
sess = sagemaker.Session()

# boto3 client
sm_boto3 = boto3.client("sagemaker")

# AWS region
region = sess.boto_session.region_name

# S3 bucket
bucket = "mobliebucket"

print("Using bucket:", bucket)
print("Region:", region)


d:\Projects\Sagemaker-project\.venv\Lib\site-packages\sagemaker_core\main\shapes.py:7104: SyntaxWarning: invalid escape sequence '\|'
  domain: The machine learning domain of the model and its components. Valid Values: COMPUTER_VISION \| NATURAL_LANGUAGE_PROCESSING \| MACHINE_LEARNING
d:\Projects\Sagemaker-project\.venv\Lib\site-packages\sagemaker_core\main\shapes.py:7909: SyntaxWarning: invalid escape sequence '\*'
  schedule_expression: A cron expression that describes details about the monitoring schedule. The supported cron expressions are:   If you want to set the job to start every hour, use the following:  Hourly: cron(0 \* ? \* \* \*)    If you want to start the job daily:  cron(0 [00-23] ? \* \* \*)    If you want to run the job one time, immediately, use the following keyword:  NOW    For example, the following are valid cron expressions:   Daily at noon UTC: cron(0 12 ? \* \* \*)    Daily at midnight UTC: cron(0 0 ? \* \* \*)    To support running every 6, 12 hours, the foll

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\aveec\AppData\Local\sagemaker\sagemaker\config.yaml


d:\Projects\Sagemaker-project\.venv\Lib\site-packages\smdebug_rulesconfig\actions\utils.py:5: SyntaxWarning: invalid escape sequence '\-'
  TRAINING_JOB_PREFIX_REGEX = "^[A-Za-z0-9\-]+$"
d:\Projects\Sagemaker-project\.venv\Lib\site-packages\smdebug_rulesconfig\actions\utils.py:6: SyntaxWarning: invalid escape sequence '\w'
  EMAIL_ADDRESS_REGEX = "^[a-z0-9]+[@]\w+[.]\w{2,3}$"
d:\Projects\Sagemaker-project\.venv\Lib\site-packages\smdebug_rulesconfig\actions\utils.py:7: SyntaxWarning: invalid escape sequence '\+'
  PHONE_NUMBER_REGEX = "^\+\d{1,15}$"


Using bucket: mobliebucket
Region: us-east-1


In [ ]:
import tensorflow as tf
print(tf.__version__)

In [1]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# -------------------------------
# Load Model
# -------------------------------
model = load_model("skin_disease_mobilenet_model.h5")

# Class labels (adjust if needed)
classes = [
    "Bacteria Infection",
    "Fungal Infection",
    "Viral Infection",
    "Parasitic Infection"
]

# -------------------------------
# Load and preprocess image
# -------------------------------
img_path ="skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (17).jpg"
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

# -------------------------------
# Make Prediction
# -------------------------------
pred = model.predict(img_array)
predicted_class = classes[np.argmax(pred)]
confidence = np.max(pred)

print("Prediction:", predicted_class)
print("Confidence:", confidence)

# -------------------------------
# Grad-CAM
# -------------------------------
last_conv_layer_name = None

# Find last convolution layer automatically
for layer in reversed(model.layers):
    if "conv" in layer.name:
        last_conv_layer_name = layer.name
        break

grad_model = tf.keras.models.Model(
    [model.inputs],
    [model.get_layer(last_conv_layer_name).output, model.output]
)

with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_array)
    class_channel = predictions[:, np.argmax(pred)]

grads = tape.gradient(class_channel, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

conv_outputs = conv_outputs[0]
heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)

# -------------------------------
# Overlay heatmap on image
# -------------------------------
img = cv2.imread(img_path)
heatmap = cv2.resize(np.array(heatmap), (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

superimposed_img = heatmap * 0.4 + img

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.title("Original Image")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Grad-CAM Explanation")
plt.imshow(cv2.cvtColor(superimposed_img.astype("uint8"), cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.show()


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'skin_disease_mobilenet_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
cd no   